In [ ]:
import os
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, BertForSequenceClassification
from pytorch_lightning import Trainer

from bertnup.data.datasets import Dnabert1Dataset
from bertnup.data.sequences import DNASequence
from bertnup.models.attention import BertNupAttention, export_bert_weights, process_attention_score
from bertnup.visualization.attention_viz import (
    visualize_token2token_scores,
    visualize_sequence_attention,
    visualize_dataset_attention,
    plot_average_attention_by_position,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from bertnup.seed import set_seed
set_seed(0)

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

In [ ]:
# Tokenizer parallelism disabled in first cell

### Loading pre-trained model and data

Download the checkpoint for a sample pre-trained model via:
- https://drive.google.com/file/d/1yz-nZ45CHNqOTbMVnuZys36Fp9Xj-NLN/view?usp=sharing

Put the checkpoint under `model_checkpoint/dnabert-1-3_pretrained.ckpt` to run the following cells.

In [ ]:
K = 3
model_name = 'armheb/DNA_bert_' + str(K)
tokenizer = AutoTokenizer.from_pretrained(model_name)
saved_model = 'pretrained_dnabert-1-3'

# Export BERT weights from checkpoint for attention extraction
export_bert_weights('model_checkpoint/dnabert-1-3_pretrained.ckpt', model_name, saved_model)
bert_model = BertForSequenceClassification.from_pretrained(saved_model, local_files_only=True, output_attentions=True)

In [ ]:
data_dir = '../Data/Stratified_K_fold_data/HS_LC/split_0/'
test_path = data_dir + 'test.csv'
test_set = Dnabert1Dataset(test_path, K)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=0)
trainer = Trainer(accelerator='auto', devices=1)

### Visualize attention scores for a single sequence

In [ ]:
sequence_idx = 10000

print('Label: ', test_set.data.label[sequence_idx])
BertNup_attn = BertNupAttention(saved_model)
final_attn_scores = BertNup_attn(
    test_set[sequence_idx]['input_ids'].view(1, -1),
    test_set[sequence_idx]['attention_mask'].view(1, -1),
).detach().numpy().reshape(1, -1)

visualize_sequence_attention(final_attn_scores, test_set.data.sequence[sequence_idx])

### Visualize attention scores for the whole test set

In [ ]:
BertNup_attn = BertNupAttention(saved_model)
attn_scores = trainer.predict(model=BertNup_attn, dataloaders=test_loader)

In [ ]:
all_attn_scores = torch.cat(attn_scores, dim=0)
processed_scores = []
for i, attn_score in enumerate(list(all_attn_scores)):
    if i % 10000 == 0:
        print(i)
    processed_scores.append(process_attention_score(attn_score, kmer=K))
final_attn_scores = np.concatenate(processed_scores, axis=0)

pos_attn = final_attn_scores[test_set.data.label == 1, :]
neg_attn = final_attn_scores[test_set.data.label == 0, :]

In [ ]:
visualize_dataset_attention(pos_attn, neg_attn)

### Compute average attention score by sequence position

In [ ]:
plot_average_attention_by_position(pos_attn, neg_attn)

### Visualize attention map for each head

In [ ]:
sequence_idx = 10
layer = 11  # layer can be in range(0,11)
head = None  # head can be in range(0,11) or None (will visualize all heads in a layer)

inputs = tokenizer.encode(str(DNASequence(test_set.data.sequence[sequence_idx]).to_kmer_sequence(K)), return_tensors='pt')
attention = bert_model(inputs)[-1]
xticks = ['CLS'] + list(test_set.data.sequence[sequence_idx].upper()) + ['SEP']
print('Label: ', test_set.data.label[sequence_idx])
output_attentions_all = torch.stack(attention)

visualize_token2token_scores(output_attentions_all[layer].squeeze().detach().cpu().numpy(), head=head, xticks=xticks)